In [41]:
import pandas as pd
import numpy as np 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor as gpr
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


In [42]:
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Resist", "Salin", "ORP", "IP", "Cor"] # 10 entradas
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "IP"] 
TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

N_COMPONENTS = 6

In [43]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []

for n in range(1, 5):
    n_data = Dataset[ Dataset["Pontos"] == f"P{n}" ]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
        
    Datasets.append(n_data)

In [44]:
def CreatePCAdf(pca):
    # Matriz de transformação do PCA
    W = pca.components_.T   # shape (n_variaveis, n_componentes)

    # Nomes das componentes
    cp_names = [f"CP{i+1}" for i in range(W.shape[1])]

    # Criar DataFrame
    df_pca = pd.DataFrame(
        data=np.round(W, 3),
        index=PREDICTORS,
        columns=cp_names
    )

    return df_pca    

In [45]:
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression

def TransformPLS(X_train, X_test, y_train):

    pls = PLSRegression(n_components=N_COMPONENTS)

    # Ajuste supervisionado (usa X e y)
    X_train_pls = pls.fit_transform(X_train, y_train)[0]
    X_test_pls  = pls.transform(X_test)

    # Variância explicada em X (aproximação)
    var_exp = np.var(X_train_pls, axis=0)
    var_ratio = var_exp / np.sum(var_exp)

    print(f"Variância (%): {np.round(var_ratio * 100, 3)}")
    print(f"Total (%): {np.round(np.sum(var_ratio) * 100, 3)}")

    # DataFrame com pesos (loadings)
    df = pd.DataFrame(
        pls.x_weights_,
        columns=[f"PLS{i+1}" for i in range(N_COMPONENTS)]
    )

    return df, pls, X_train_pls, X_test_pls


In [46]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

def plot_train_test_samples(
    y_train, y_train_pred,
    y_test, y_test_pred,
    title="GPR – Treinamento e Teste"
):

    samples_train = np.arange(len(y_train))
    samples_test = np.arange(len(y_test))

    # Figura
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharey=True)

    # -------- Subfigura (a): Treinamento --------
    axes[0].plot(
        samples_train, y_train,
        'o', label="y train (real)", markersize=5
    )
    axes[0].plot(
        samples_train, y_train_pred,
        'x', label="y train (pred)", markersize=5
    )

    axes[0].set_title("(a) Treinamento")
    axes[0].set_xlabel("Amostras")
    axes[0].set_ylabel("Valor")
    axes[0].legend()
    axes[0].grid(True)

    # axes[0].text(
    #     0.02, 0.95,
    #     f"MSE = {mse_train:.4f}\n$R^2$ = {r2_train:.4f}",
    #     transform=axes[0].transAxes,
    #     verticalalignment="top",
    #     bbox=dict(boxstyle="round", alpha=0.85)
    # )

    # -------- Subfigura (b): Teste --------
    axes[1].plot(
        samples_test, y_test,
        'o', label="y test (real)", markersize=5
    )
    axes[1].plot(
        samples_test, y_test_pred,
        'x', label="y test (pred)", markersize=5
    )

    axes[1].set_title("(b) Teste")
    axes[1].set_xlabel("Amostras")
    axes[1].set_ylabel("Valor")
    axes[1].legend()
    axes[1].grid(True)

    # axes[1].text(
    #     0.02, 0.95,
    #     f"MSE = {mse_test:.4f}\n$R^2$ = {r2_test:.4f}",
    #     transform=axes[1].transAxes,
    #     verticalalignment="top",
    #     bbox=dict(boxstyle="round", alpha=0.85)
    # )

    fig.suptitle(title, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [47]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

def ComputeMetrics(y_train, y_train_pred, y_test, y_test_pred):
     return {
        "mse_train": mean_squared_error(y_train, y_train_pred),
        "r2_train":  r2_score(y_train, y_train_pred),
        "mse_test":  mean_squared_error(y_test, y_test_pred),
        "r2_test":   r2_score(y_test, y_test_pred)
    }

In [48]:
GPR_PARAMS = {
    "Fe": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Al": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "As": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Pb": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "Zn": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "Hg": {"nu": 0.5, "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Co": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e10, "alpha": 1e-3},
    "V":  {"nu": 0.25, "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Ba": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Mn": {"nu": 1.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
}

In [49]:


def GprModel(X_train_pca, X_test_pca, y_train, y_test, target):

    params = GPR_PARAMS[target]

    kernel = C(1.0) * Matern(
        length_scale=np.ones(X_train_pca.shape[1]),
        nu=params["nu"],
        length_scale_bounds=(params["ls_min"], params["ls_max"])
    )

    model = gpr(
        kernel=kernel,
        alpha=params["alpha"],
        normalize_y=True,
        n_restarts_optimizer=20
    )

   # Treinamento
    model.fit(X_train_pca, y_train)

    # Predições
    y_train_pred = model.predict(X_train_pca)
    y_test_pred = model.predict(X_test_pca)

    # Desnormalização
    y_train_denorm = OUT_SCALER.inverse_transform(y_train.reshape(-1, 1)).ravel()
    y_train_pred_denorm = OUT_SCALER.inverse_transform(y_train_pred.reshape(-1, 1)).ravel()

    y_test_denorm = OUT_SCALER.inverse_transform(y_test.reshape(-1, 1)).ravel()
    y_test_pred_denorm = OUT_SCALER.inverse_transform(y_test_pred.reshape(-1, 1)).ravel()
    
    metrics = ComputeMetrics(y_train_denorm, y_train_pred_denorm,  y_test_denorm, y_test_pred_denorm)
    
    return  metrics

    # plot_train_test_samples(
    #     y_train_denorm, y_train_pred_denorm,
    #     y_test_denorm, y_test_pred_denorm,
    #     title="GPR com Kernel Matern (ν = 0.5)"
    # )

In [ ]:
Results = {}

for i, Dataset in enumerate(Datasets):
    print(f"++++++++++++++++++++++ Pontos {i} ++++++++++++++++++++++++++")

    X = Dataset[PREDICTORS].values
    Y = Dataset[TARGETS].values
    
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

    X_train_scaled = SCALER.fit_transform(X_train)
    X_test_scaled  = SCALER.transform(X_test)
    
    for j, target in enumerate(TARGETS):
        
        print(f" → {target}")      
        
        y_train = Y_train[:, j]
        y_test  = Y_test[:, j]
        
        y_train = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()
        
        df, pls, X_train_pls, X_test_pls = TransformPLS(X_train_scaled, X_test_scaled, y_train)
        # display(df)
        
        metrics = GprModel(X_train_pls, X_test_pls, y_train, y_test, target)
        
        Results[target] = metrics
        
    display(pd.DataFrame(Results).T)


++++++++++++++++++++++ Pontos 0 ++++++++++++++++++++++++++
 → Fe
Variância (%): [2.3445e+01 4.1894e+01 1.2567e+01 5.1690e+00 1.6924e+01 2.0000e-03]
Total (%): 100.0
 → Al
Variância (%): [3.3672e+01 3.3418e+01 5.7710e+00 1.0999e+01 1.6139e+01 2.0000e-03]
Total (%): 100.0


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 1000000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k2__length_scale is close to the specified upper bound 1000000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


 → As
Variância (%): [2.0117e+01 4.4939e+01 6.6250e+00 1.6166e+01 1.2150e+01 2.0000e-03]
Total (%): 100.0


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 1000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 1000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k2__length_scale is close to the specified upper bound 1000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib

 → Pb
Variância (%): [4.5553e+01 1.7666e+01 1.1305e+01 7.0530e+00 1.8420e+01 2.0000e-03]
Total (%): 100.0


...

 → Zn
Variância (%): [41.985 21.435 20.455  3.202 12.923  0.   ]
Total (%): 100.0
 → Hg
Variância (%): [22.523 41.45  17.095  2.781 16.152  0.   ]
Total (%): 100.0
 → Co
Variância (%): [19.585 36.005 24.868  3.725 15.816  0.   ]
Total (%): 100.0
 → V
Variância (%): [25.972 41.088  9.235  7.021 16.684  0.   ]
Total (%): 100.0
 → Ba
Variância (%): [45.672 18.491 20.192  2.748 12.896  0.   ]
Total (%): 100.0


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k2__length_scale is close to the specified upper bound 1000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


 → Mn
Variância (%): [30.085 31.2   20.296  2.88  15.539  0.   ]
Total (%): 100.0


,mse_train,r2_train,mse_test,r2_test
Fe,4.980257e-01,0.999997,262707.137923,0.095870
Al,7.261680e-02,0.999998,112705.026670,-56.403258
As,1.725259e-08,0.999997,0.000382,0.800611
Pb,5.799590e-06,0.999994,0.072038,-5.275402
Zn,1.109052e-03,0.999997,481.071811,-0.057589
Hg,6.700892e-09,0.999999,0.007058,-0.007351
Co,1.057155e-07,0.999995,0.019318,0.076688
V,1.547366e-07,0.999998,0.032728,0.138711
Ba,8.162890e-04,0.999996,72.757715,-0.366592
Mn,9.231544e-03,0.999990,1195.084209,0.095073


: 